In [50]:
# Load libraries
library("anndata")
library("DESeq2")
library("reticulate")

# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

# Import scanpy
sc <- import("scanpy")

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [ ]:
# Read in the adata
adata <- sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/postprocess_adata.020624/adata.combined.postprocess.h5ad')

In [ ]:
# Filter rows for primary tumor
primary_tumor <- adata[rownames(adata$obs) %in% '146P', ]
write.h5ad(primary_tumor, '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/primary_tumor.h5ad')

# Filter rows for metastatic tumor
metastatic_tumor <- adata[rownames(adata$obs) %in% 'KG146Li', ]
write.h5ad(metastatic_tumor, '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/metastatic_tumor.h5ad')

In [ ]:
adata$obs

In [ ]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
#count_matrix <- adata$X
#count_matrix <- as(count_matrix, "matrix")
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [ ]:
count_matrix

In [ ]:
# Extract sample names from row names
sample_names <- rownames(adata$X)

# Assuming 'adata' is your AnnData object

# Extract sample names from row names
sample_names <- rownames(adata$X)

# Create a new column 'Tumor_Site' based on sample names
adata$obs$Tumor_Site <- ifelse(grepl("146P", sample_names), "Primary", 
                               ifelse(grepl("146Li", sample_names), "Metastatic", NA))

# Create a new column 'Culture_Media' based on sample names
adata$obs$Culture_Media <- ifelse(grepl("BASE", sample_names, ignore.case = TRUE), "BASE", 
                                  ifelse(grepl("HISC", sample_names, ignore.case = TRUE), "HISC",
                                  ifelse(grepl("dedifferentiation", sample_names, ignore.case = TRUE), "Dedifferentiated", "Unknown")))

# Create a new column 'ZFP_Expression' based on sample names
adata$obs$ZFP_Expression <- ifelse(grepl("CTRL", sample_names, ignore.case = TRUE), "CTRL", 
                                  ifelse(grepl("ZFP", sample_names, ignore.case = TRUE), "ZFPKD", "Unknown"))

In [ ]:
adata$obs

In [ ]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression")]

# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)

# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression
)

In [ ]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

# Set the reference levels
coldata$Tumor_Site<-relevel(coldata$Tumor_Site,ref="Primary")
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression 
)

In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/sc_zfpexp_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/sc_zfpexp_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/sc_zfpexp_results.rds", sep=""))


### DEGs for Primary Tumor

In [51]:
# Read in the adata for primary tumor site
adata_primary = sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/input/degs/primary_tumor.h5ad')

In [ ]:
adata_primary

In [52]:
# Get the counts matrix
counts = as.matrix(adata_primary$X)

# Normalize
ms <- rowSums(counts)
norm_df <- counts/ms * median(ms)

# Log transform
counts <- log(norm_df + 1)

# Add pseudocount
counts <- counts + 1

# Round since error that not all values are integers
counts <- round(counts)

# Transpose
counts <- t(counts)

# Assign row and column names to the counts matrix
rownames(counts) <- adata_primary$var_names
colnames(counts) <- adata_primary$obs_names

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 7.3 GiB”


In [53]:
# Test out with a small random subset
ind = sample.int(ncol(counts), 500)

counts = counts[,ind]

In [ ]:
# Define the column data
coldata <- adata_primary$obs[ind ,c("Culture_Media", "ZFP_Expression")]

# Convert column data to factors
coldata$Culture_Media <- factor(coldata$Culture_Media, levels = c("BASE","Dedifferentiated", "HISC"))
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression, levels = c("ZFP_KD","CTRL"))

# Set the reference levels
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

In [54]:
coldata$group <- factor(paste0(adata_primary$obs[ind ,c("Culture_Media")], adata_primary$obs[ind ,c("ZFP_Expression")]))

In [56]:
# Set the reference levels
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

In [57]:
coldata

,Culture_Media,ZFP_Expression,group
,<fct>,<fct>,<fct>
146P_BASE_shCTRL_TCCAGAAAGTGTTCCA-1,BASE,CTRL,HISCZFP_KD
146P_BASE_shZFP36L2_4_TTGCATTCATGGAATA-1,BASE,ZFP_KD,BASEZFP_KD
146P_BASE_shZFP36L2_4_CACAACAAGTACGAGC-1,BASE,ZFP_KD,DedifferentiatedZFP_KD
146P_BASE_shZFP36L2_4_GGGAGATAGTCTAACC-1,BASE,ZFP_KD,DedifferentiatedCTRL
146P_BASE_shZFP36L2_4_GGTGTTAAGCGAGTCA-1,BASE,ZFP_KD,BASEZFP_KD
146P_BASE_shZFP36L2_4_TGAATGCTCCGACATA-1,BASE,ZFP_KD,HISCZFP_KD
146P_BASE_shZFP36L2_3_TGAGCATAGTTCGCAT-1,BASE,ZFP_KD,BASEZFP_KD
146P_dedifferentiation_shZFP36L2_4_TACAGGTTCCTTATGT-1,Dedifferentiated,ZFP_KD,BASEZFP_KD
146P_BASE_shCTRL_CAGAGCCGTGCACGCT-1,BASE,CTRL,HISCZFP_KD


In [59]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = counts,
  colData = coldata,
  design = ~ group 
)

ERROR: Error in SummarizedExperiment(assays = SimpleList(counts = countData), : the rownames and colnames of the supplied assay(s) must be NULL or
  identical to those of the SummarizedExperiment object (or derivative)
  to construct


In [ ]:
dds_1$group

In [ ]:
# Filter the DDS object
keep <- rowSums(counts(dds_1)) >= 10
dds_1 <- dds_1[keep,]

In [ ]:
# Use grouping for interaction term
dds_1$group <- factor(paste0(dds_1$Culture_Media, dds_1$ZFP_Expression))
design(dds_1) <- ~ group

In [ ]:
# Estimate size factors
dds_1 <- estimateSizeFactors(dds_1)

In [ ]:
# Estimate dispersions
dds_1 <- estimateDispersionsGeneEst(dds_1)

In [ ]:
# Get dispersions
dispersions(dds_1) <- mcols(dds_1)$dispGeneEst

In [ ]:
# Perform binomial Wald test
dds_1 <- nbinomWaldTest(dds_1)

In [ ]:
# Get results
resultsNames(dds_1)

### 

In [ ]:
dds_1

In [ ]:
dispersions(dds_1) <- mcols(dds_1)$dispGeneEst

In [ ]:
dds_1 <- nbinomWaldTest(dds_1)

In [ ]:
resultsNames(dds_1)

In [ ]:
results(dds_1, contrast=c("group", "DifferentiatedZFP_KD", "DifferentiatedCTRL"))

In [ ]:
results(dds_1)

In [ ]:
# Extract count matrix from the raw layer
count_matrix <- as.matrix(adata_primary$X)
# Transpose the ocunt matrix
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)
# Add a pseudocount
count_matrix <- count_matrix + 1 
# Normalize the count matrix by library size
library_sizes <- colSums(count_matrix)
count_matrix <- count_matrix / library_sizes * mean(library_sizes)

In [ ]:
counts

In [ ]:
# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = counts,
  colData = coldata,
  design = ~ Culture_Media + ZFP_Expression 
)

In [ ]:
# Add a pseudo-count of 1 to all counts
counts <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata_primary$var_names

# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Culture_Media + ZFP_Expression:Culture_Media 
)

In [ ]:
dds_1 <- estimateSizeFactors(dds_1)

In [ ]:
dds_1 <- estimateDispersionsGeneEst(dds_1)

In [ ]:
# Perform DESeq analysis
dds_1 <- DESeq(dds_1)

# Extract results
result_names_1 <- resultsNames(dds_1)
result_names_1

# Specify the contrast and extract results
res_1 <- results(dds_1)
res_1

In [ ]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/sc_zfpexp_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/sc_zfpexp_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/sc_zfpexp_results.rds", sep=""))